# BTC 5-Minute Direction Prediction

Binary classification project predicting whether BTC-USD will be higher five minutes ahead using 1-minute market data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier

## Data

In [ ]:
df = yf.download(
    "BTC-USD",
    period="5d",
    interval="1m",
    auto_adjust=False
)

print("Shape:", df.shape)
df.head()

Shape: (6847, 6)


In [ ]:
df.columns = df.columns.get_level_values(0)
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

## Target

In [ ]:
df["future_close_5m"] = df["close"].shift(-5)

model_df = df.dropna(subset=["future_close_5m"]).copy()

model_df["target"] = (
    model_df["future_close_5m"] > model_df["close"]
).astype(int)

model_df[["close", "future_close_5m", "target"]].head(10)

`target = 1` if BTC is higher five minutes later, otherwise `0`.  
`future_close_5m` is used only to create the label and is not used as a model feature.

## Feature Engineering

In [ ]:
model_df["close_lag_1"] = model_df["close"].shift(1)
model_df["close_lag_2"] = model_df["close"].shift(2)
model_df["close_lag_5"] = model_df["close"].shift(5)

model_df["close_rolling_mean_5"] = (
    model_df["close"]
    .rolling(5)
    .mean()
)

model_df["price_change_1m"] = (
    model_df["close"] - model_df["close_lag_1"]
)

In [ ]:
model_df["return_1m"] = model_df["close"].pct_change()

model_df["return_lag_1"] = model_df["return_1m"].shift(1)
model_df["return_lag_2"] = model_df["return_1m"].shift(2)
model_df["return_lag_5"] = model_df["return_1m"].shift(5)

model_df["return_rolling_mean_5"] = (
    model_df["return_1m"]
    .rolling(5)
    .mean()
)

return_features = [
    "return_1m",
    "return_lag_1",
    "return_lag_2",
    "return_lag_5",
    "return_rolling_mean_5",
]

return_df = model_df.dropna(subset=return_features).copy()

X_return = return_df[return_features]
y_return = return_df["target"]

split_idx = int(len(return_df) * 0.8)

X_train_r = X_return.iloc[:split_idx]
X_test_r = X_return.iloc[split_idx:]

y_train_r = y_return.iloc[:split_idx]
y_test_r = y_return.iloc[split_idx:]

## Logistic Regression Baselines

In [ ]:
price_features = [
    "close",
    "close_lag_1",
    "close_lag_2",
    "close_lag_5",
    "close_rolling_mean_5",
    "price_change_1m",
]

price_df = model_df.dropna(subset=price_features).copy()

X_price = price_df[price_features]
y_price = price_df["target"]

In [ ]:
split_idx = int(len(price_df) * 0.8)

X_train = X_price.iloc[:split_idx]
X_test = X_price.iloc[split_idx:]

y_train = y_price.iloc[:split_idx]
y_test = y_price.iloc[split_idx:]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target proportions:")
print(y_train.value_counts(normalize=True))
print("\nTest target proportions:")
print(y_test.value_counts(normalize=True))

Train shape: (5469, 6)
Test shape: (1368, 6)

Train target proportions:
target
0    0.535381
1    0.464619
Name: proportion, dtype: float64

Test target proportions:
target
0    0.537281
1    0.462719
Name: proportion, dtype: float64


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logistic_price = LogisticRegression()
logistic_price.fit(X_train_scaled, y_train)

y_pred = logistic_price.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nPrediction counts:")
print(pd.Series(y_pred).value_counts())
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5314327485380117

Prediction counts:
0    1342
1      26
Name: count, dtype: int64

Confusion matrix:
[[718  17]
 [624   9]]


In [ ]:
model_df["return_1m"] = model_df["close"].pct_change()

model_df["return_lag_1"] = model_df["return_1m"].shift(1)
model_df["return_lag_2"] = model_df["return_1m"].shift(2)
model_df["return_lag_5"] = model_df["return_1m"].shift(5)

model_df["return_rolling_mean_5"] = (
    model_df["return_1m"]
    .rolling(5)
    .mean()
)

return_features = [
    "return_1m",
    "return_lag_1",
    "return_lag_2",
    "return_lag_5",
    "return_rolling_mean_5",
]

return_df = model_df.dropna(subset=return_features).copy()

X_return = return_df[return_features]
y_return = return_df["target"]

split_idx = int(len(return_df) * 0.8)

X_train_r = X_return.iloc[:split_idx]
X_test_r = X_return.iloc[split_idx:]

y_train_r = y_return.iloc[:split_idx]
y_test_r = y_return.iloc[split_idx:]

In [ ]:
scaler_r = StandardScaler()

X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

logistic_return = LogisticRegression()
logistic_return.fit(X_train_r_scaled, y_train_r)

y_pred_r = logistic_return.predict(X_test_r_scaled)

print("Accuracy:", accuracy_score(y_test_r, y_pred_r))
print("\nPrediction counts:")
print(pd.Series(y_pred_r).value_counts())
print("\nTest target proportions:")
print(y_test_r.value_counts(normalize=True))
print("\nConfusion matrix:")
print(confusion_matrix(y_test_r, y_pred_r))

Accuracy: 0.5332845647403073

Prediction counts:
0    1350
1      17
Name: count, dtype: int64

Test target proportions:
target
0    0.536942
1    0.463058
Name: proportion, dtype: float64

Confusion matrix:
[[723  11]
 [627   6]]


## XGBoost

In [ ]:
xgb_return = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

xgb_return.fit(X_train_r, y_train_r)

y_pred_xgb = xgb_return.predict(X_test_r)

print("XGBoost accuracy:", accuracy_score(y_test_r, y_pred_xgb))
print("\nPrediction counts:")
print(pd.Series(y_pred_xgb).value_counts())

XGBoost accuracy: 0.5303584491587418

Prediction counts:
0    1264
1     103
Name: count, dtype: int64


In [ ]:
combined_features = [
    "close",
    "close_lag_1",
    "close_lag_2",
    "close_lag_5",
    "close_rolling_mean_5",
    "price_change_1m",
    "return_1m",
    "return_lag_1",
    "return_lag_2",
    "return_lag_5",
    "return_rolling_mean_5",
]

combined_df = model_df.dropna(subset=combined_features).copy()

X_combined = combined_df[combined_features]
y_combined = combined_df["target"]

split_idx = int(len(combined_df) * 0.8)

X_train_c = X_combined.iloc[:split_idx]
X_test_c = X_combined.iloc[split_idx:]

y_train_c = y_combined.iloc[:split_idx]
y_test_c = y_combined.iloc[split_idx:]

In [ ]:
xgb_combined = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

xgb_combined.fit(X_train_c, y_train_c)

y_pred_c = xgb_combined.predict(X_test_c)

print("Combined XGBoost accuracy:", accuracy_score(y_test_c, y_pred_c))
print("\nPrediction counts:")
print(pd.Series(y_pred_c).value_counts())

Combined XGBoost accuracy: 0.5398683247988295

Prediction counts:
0    1059
1     308
Name: count, dtype: int64


## Leakage-Aware Evaluation

Because the target looks five observations ahead, a 5-row gap is kept between training and evaluation periods so training labels do not overlap the validation window.

In [ ]:
split_idx = int(len(combined_df) * 0.8)
gap = 5

X_train_gap = X_combined.iloc[:split_idx - gap]
y_train_gap = y_combined.iloc[:split_idx - gap]

X_test_gap = X_combined.iloc[split_idx:]
y_test_gap = y_combined.iloc[split_idx:]

xgb_gap = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

xgb_gap.fit(X_train_gap, y_train_gap)

y_pred_gap = xgb_gap.predict(X_test_gap)

print("Gap XGBoost accuracy:", accuracy_score(y_test_gap, y_pred_gap))
print("\nPrediction counts:")
print(pd.Series(y_pred_gap).value_counts())
print("\nConfusion matrix:")
print(confusion_matrix(y_test_gap, y_pred_gap))

Gap XGBoost accuracy: 0.5362106803218727

Prediction counts:
0    1070
1     297
Name: count, dtype: int64

Confusion matrix:
[[585 149]
 [485 148]]


## Walk-Forward Validation

`TimeSeriesSplit` is used with expanding training windows, five validation folds, and a 5-row gap.

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=5,
    gap=5
)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(X_combined),
    start=1
):
    print(
        f"Fold {fold}: "
        f"train rows={len(train_idx)}, "
        f"validation rows={len(val_idx)}"
    )

Fold 1: train rows=1136, validation rows=1138
Fold 2: train rows=2274, validation rows=1138
Fold 3: train rows=3412, validation rows=1138
Fold 4: train rows=4550, validation rows=1138
Fold 5: train rows=5688, validation rows=1138


In [ ]:
fold_accuracies = []
fold_baselines = []

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(X_combined),
    start=1
):
    X_train_fold = X_combined.iloc[train_idx]
    X_val_fold = X_combined.iloc[val_idx]

    y_train_fold = y_combined.iloc[train_idx]
    y_val_fold = y_combined.iloc[val_idx]

    model_fold = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )

    model_fold.fit(X_train_fold, y_train_fold)
    y_pred_fold = model_fold.predict(X_val_fold)

    accuracy = accuracy_score(y_val_fold, y_pred_fold)

    baseline = (
        y_val_fold
        .value_counts(normalize=True)
        .max()
    )

    fold_accuracies.append(accuracy)
    fold_baselines.append(baseline)

    print(
        f"Fold {fold}: "
        f"Model = {accuracy:.4f} | "
        f"Baseline = {baseline:.4f}"
    )

Fold 1: Model = 0.4789 | Baseline = 0.5316
Fold 2: Model = 0.4482 | Baseline = 0.5518
Fold 3: Model = 0.5141 | Baseline = 0.5149
Fold 4: Model = 0.5018 | Baseline = 0.5360
Fold 5: Model = 0.5299 | Baseline = 0.5395


In [ ]:
mean_accuracy = np.mean(fold_accuracies)
mean_baseline = np.mean(fold_baselines)

print("Average model accuracy:", mean_accuracy)
print("Average baseline:", mean_baseline)

Average model accuracy: 0.49455184534270646
Average baseline: 0.5347978910369069


## Results

| Model / Evaluation | Accuracy |
|---|---:|
| Logistic Regression — price features | 53.14% |
| Logistic Regression — return features | 53.33% |
| XGBoost — return features | 53.04% |
| XGBoost — combined features, single split | 53.99% |
| XGBoost — combined features with 5-row gap | 53.62% |
| **XGBoost — 5-fold walk-forward average** | **49.46%** |
| **Walk-forward majority baseline** | **53.48%** |

## Conclusion

The tested features did not produce a robust predictive signal for BTC's 5-minute direction. The final walk-forward XGBoost accuracy was below the majority-class baseline.

The project demonstrates leakage-aware time-series evaluation using lagged features, rolling features, chronological splits, a forecast-horizon gap, and walk-forward validation.

## Limitations

- Only about five days of 1-minute BTC-USD data were used.
- The dataset does not include exchange-level order-book or trade-flow features.
- Transaction costs, slippage, latency, and execution are not modeled.
- A longer dataset and richer market microstructure features would be required before drawing trading conclusions.